### loading dataset and preprocessing

In [1]:
import re
import emoji
import contractions
from nltk.stem import WordNetLemmatizer
import spacy
from nltk.corpus import stopwords
import nltk
import numpy as np
# Download stopwords if not already
nltk.download("stopwords")
nltk.download("wordnet")

# Load spaCy
nlp = spacy.load("en_core_web_sm")

stop_words = set(stopwords.words("english"))
lemmatizer = WordNetLemmatizer()

def text_preprocessing(text):
    text = emoji.demojize(text)
    text = contractions.fix(text)
    text = text.lower()
    text = re.sub(r'\d{4}-\d{2}-\d{2}', 'DATE', text)
    text = re.sub(r'\d+', "NUM", text)
    text = re.sub(r'\b\w+@\w+\.\w+\b', 'EMAIL', text)
    text = re.sub(r'[^\w\s]', "", text)
    # Tokenize with spaCy
    doc = nlp(text)
    #print([token.text for token in doc])
    # Lemmatize + remove stopwords
    tokens = [
        lemmatizer.lemmatize(token.text) 
        for token in doc 
        if token.text not in stop_words and not token.is_space
    ]

    return  tokens #" ".join(tokens)


[nltk_data] Downloading package stopwords to
[nltk_data]     /home/mudasir/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /home/mudasir/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


In [33]:
spam_data = {}

with open("/home/mudasir/ankit/NLP/Lab_06/spam.txt" , 'r') as f:
    text = []
    label = []
    for line in f:
        label.append(line.split()[-1])
        text.append(" ".join(line.split()[:-1]))
    spam_data['text'] = text
    spam_data['label'] = label

In [34]:
print(spam_data['text'][:5])
print(spam_data['label'][:5])

['Go until jurong point, crazy.. Available only in bugis n great world la e buffet... Cine there got amore wat...', 'Ok lar... Joking wif u oni...', 'U dun say so early hor... U c already then say...', "Nah I don't think he goes to usf, he lives around here though", 'Even my brother is not like to speak with me. They treat me like aids patent.']
['0', '0', '0', '0', '0']


In [35]:
preprocessed_spam_data = {}
preprocessed_spam_data['text'] = [text_preprocessing(t) for t in spam_data['text']]
preprocessed_spam_data['label'] = spam_data['label']


In [36]:
print(preprocessed_spam_data['text'][:5])
print(preprocessed_spam_data['label'][:5]) 


[['go', 'jurong', 'point', 'crazy', 'available', 'bugis', 'n', 'great', 'world', 'la', 'e', 'buffet', 'cine', 'got', 'amore', 'wat'], ['ok', 'lar', 'joking', 'wif', 'oni'], ['dun', 'say', 'early', 'hor', 'c', 'already', 'say'], ['nah', 'think', 'go', 'usf', 'life', 'around', 'though'], ['even', 'brother', 'like', 'speak', 'treat', 'like', 'aid', 'patent']]
['0', '0', '0', '0', '0']


In [59]:
from torch.utils.data import Dataset, random_split

class SpamDataset(Dataset):
    def __init__(self, texts, labels):
        self.texts = texts
        self.labels = labels
    
    def __len__(self):
        return len(self.texts)
    
    def __getitem__(self, idx):
        return self.texts[idx], self.labels[idx]

# Create dataset
dataset = SpamDataset(preprocessed_spam_data['text'], preprocessed_spam_data['label'])

# Split sizes
dataset_len = len(dataset)
train_len = int(dataset_len * 0.7)
val_len = int(dataset_len * 0.2)
test_len = dataset_len - train_len - val_len  # ensure total matches

# Split
train_dataset, val_dataset, test_dataset = random_split(dataset, [train_len, val_len, test_len])
print(f"Train size: {len(train_dataset)}, Val size: {len(val_dataset)}, Test size: {len(test_dataset)}")
print("Tain, Val, Test ratio:" , len(train_dataset)/dataset_len, len(val_dataset)/dataset_len, len(test_dataset)/dataset_len)

Train size: 1082, Val size: 309, Test size: 156
Tain, Val, Test ratio: 0.6994182288299935 0.1997414350355527 0.10084033613445378


---

###  what do we require from lab 04?
1. word2vec model target_embedding layer
2. input to this layer is the index value of each word
3. mapping of word->index is given by word2idx() dictionary
4. word2idx() requires lab 04 dataset and creating vocab fromn it and then creating word2idx() dictionary
5. to handle oov add <|pad|> token to voacb and then create word2idx , then train word2vec algo

In [ ]:
corpus.append("<pad>")  # to handle oov
word_counts = Counter(corpus)
vocab = sorted(word_counts, key=word_counts.get, reverse=True)
word2idx = {w: idx for idx, w in enumerate(vocab)}
idx2word = {idx: w for w, idx in word2idx.items()}
vocab_size = len(vocab)

print("Vocab size:", vocab_size)

In [57]:
# getting pretrained embeddings from the saved word2vec model
import sys
sys.path.append("/home/mudasir/ankit/NLP")
from custom_word2vec import SkipGramNegSampling  # class must be importable

model = torch.load("/home/mudasir/ankit/NLP/Lab_04/Ankit_model.pth", weights_only=False)


In [ ]:
pretrained_embeddings = model.target_embeddings

### RNN model

In [56]:
import torch 
import torch.nn as nn


class RNNModel(nn.Module):
    def __init__(self, hidden_size, output_size, vocab_size=None, embed_size=None,n_layers=1, bidirectional=False, pretrained_embeddings=None):
        super(RNNModel, self).__init__()
        if pretrained_embeddings is not None:
            vocab_size, embed_size = pretrained_embeddings.weight.shape
            self.embedding = nn.Embedding.from_pretrained(pretrained_embeddings.weight, freeze=False)
        else:
            self.embedding = nn.Embedding(vocab_size, embed_size)

        self.rnn = nn.RNN(embed_size, hidden_size, num_layers=n_layers,bidirectional=bidirectional, batch_first=True)
        self.fc = nn.Linear(hidden_size * (2 if bidirectional else 1), output_size)
        self.softmax = nn.LogSoftmax(dim=1)

    def forward(self, x):
        x = self.embedding(x)
        rnn_out, _ = self.rnn(x)
        out = rnn_out[:, -1, :]  # last time step
        out = self.fc(out)
        out = self.softmax(out)
        return out


In [ ]:
def custom_collate_fn(batch):
    texts, labels = zip(*batch)
    lengths = [len(text) for text in texts]
    max_length = max(lengths)
    
    padded_texts = []
    for text in texts:
        padded_text = torch.cat([text, torch.zeros(max_length - len(text), dtype=torch.long)])
        padded_texts.append(padded_text)
    
    return torch.stack(padded_texts), torch.tensor(labels), torch.tensor(lengths)

In [ ]:
# Train, test, val dataloaders
from torch.utils.data import DataLoader
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, collate_fn=lambda x: x)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, collate_fn=lambda x: x)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, collate_fn=lambda x: x)